In [2]:
import sys
import os

# Go up one level to the main project directory and add it to Python's path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [12]:
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

In [13]:

import os
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

# Import the preprocessing function you just created
from src.preprocess import preprocess_data



# 1. Load data
print("Loading training data...")
# train_path = os.path.join(data_dir, 'train.csv')
train_data = pd.read_csv(r"..\data\raw\train.csv")

# 2. Preprocess
print("Preprocessing training data...")
df = preprocess_data(train_data)

# ==========================================
# PHASE 1: LOCAL VALIDATION
# ==========================================
print("\n--- PHASE 1: Local Validation ---")
X = df.drop('health_condition', axis=1)
y = df['health_condition']

# Split into Local Train/Test
X_train_local, X_test_local, y_train_local, y_test_local = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

local_model = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss')
print("Training local model...")
local_model.fit(X_train_local, y_train_local)

local_preds = local_model.predict(X_test_local)
print("Local Validation Score:\n")
print(classification_report(y_test_local, local_preds))
    


Loading training data...
Preprocessing training data...

--- PHASE 1: Local Validation ---
Training local model...
Local Validation Score:

              precision    recall  f1-score   support

           0       0.97      0.99      0.98    118512
           1       0.95      0.80      0.87      7961
           2       0.96      0.78      0.86     11545

    accuracy                           0.97    138018
   macro avg       0.96      0.86      0.90    138018
weighted avg       0.97      0.97      0.96    138018



In [14]:
from sklearn.utils.class_weight import compute_sample_weight

In [15]:
train_data = pd.read_csv(r"..\data\raw\train.csv")
df = preprocess_data(train_data)

X = df.drop('health_condition', axis=1)
y = df['health_condition']

# Split into Local Train/Test
X_train_local, X_test_local, y_train_local, y_test_local = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

weights_local = compute_sample_weight(class_weight='balanced', y=y_train_local)

local_model = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss')
print("Training local model...")

local_model.fit(X_train_local, y_train_local, sample_weight=weights_local)
local_preds = local_model.predict(X_test_local)
print("Local Validation Score:\n")
print(classification_report(y_test_local, local_preds))


Training local model...
Local Validation Score:

              precision    recall  f1-score   support

           0       0.99      0.89      0.93    118512
           1       0.54      0.91      0.68      7961
           2       0.60      0.93      0.73     11545

    accuracy                           0.89    138018
   macro avg       0.71      0.91      0.78    138018
weighted avg       0.93      0.89      0.90    138018



In [ ]:
# src_dir = os.path.dirname(os.path.abspath(__file__))
# project_dir = os.path.abspath(os.path.join(src_dir, '..'))
# results_dir = os.path.join(project_dir, 'results')
# data_dir = os.path.join(project_dir, 'data', 'raw')

def get_next_filename(base_dir, base_name="submission", ext=".csv"):
    """
    Finds the next available filename to avoid overwriting old submissions.
    Example: submission_1.csv, submission_2.csv
    """
    counter = 1
    while True:
        filename = os.path.join(base_dir, f"{base_name}_{counter}{ext}")
        if not os.path.exists(filename):
            return filename
        counter += 1

print("\n--- PHASE 2: Final Predictions ---")
# Retrain on 100% of the Training Data
weights_final = compute_sample_weight(class_weight='balanced', y=y)
final_model = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss')
print("Retraining model on ALL training data...")

final_model.fit(X, y, sample_weight=weights_final)

print("Loading and preprocessing test data...")
# test_path = os.path.join(data_dir, 'test.csv')
raw_test_df = pd.read_csv(r"..\data\raw\test.csv")
passenger_ids = raw_test_df['id']

# Preprocess test data
clean_test_df = preprocess_data(raw_test_df)

# Align columns perfectly with training data
X_test_kaggle = clean_test_df.reindex(columns=X.columns, fill_value=0)

print("Generating predictions...")
kaggle_preds = final_model.predict(X_test_kaggle)

# Format submission
submission = pd.DataFrame({
    'id': passenger_ids,
    'health_condition': kaggle_preds
})

# Map numeric predictions (0, 1, 2) back to text labels for Kaggle
reverse_mapping = {0: 'at-risk', 1: 'fit', 2: 'unhealthy'}
submission['health_condition'] = submission['health_condition'].map(reverse_mapping)

# Get the next incremental filename and save
save_path = r"../results/submission_3.csv"
submission.to_csv(save_path, index=False)
print(f"\nSuccess! Submission saved at: {save_path}")


--- PHASE 2: Final Predictions ---
Retraining model on ALL training data...
Loading and preprocessing test data...
Generating predictions...

Success! Submission saved at: ../results/submission_3.csv
